In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import pyfixest as pf

import warnings

warnings.filterwarnings(
    "ignore",
    message=r".*singleton fixed effect\(s\) dropped from the model.*",
    category=UserWarning,
)

In [2]:
population = pd.read_parquet("../../data/population/population.parquet")

population_total = (
    population
        .groupby(["mun_id", "year"])
        .agg(total_population = ("population", "sum"))
        .reset_index()
)

fertility_groups = [
    '15_19', '20_24', '25_29', '30_34', '35_39', '40_44'
]

population_fertility = (
    population
        .loc[population["age_group"].isin(fertility_groups)]
        .loc[population["sex"] == "female"]
        .groupby(["mun_id", "year"])
        .agg(fertile_population = ("population", "sum"))
        .reset_index()
)

population = pd.merge(
    population_total, 
    population_fertility,
    on = ["mun_id", "year"], how = "outer"
)

population

,mun_id,year,total_population,fertile_population
0,110001,2000,27095,6451
1,110001,2001,26827,6417
2,110001,2002,26132,6356
3,110001,2003,26352,6372
4,110001,2004,26134,6356
...,...,...,...,...
144678,530010,2021,3094325,797702
144679,530010,2022,2952426,736448
144680,530010,2023,2967543,729914
144681,530010,2024,2982818,722642


In [3]:
birth_weight = (
    pd.read_parquet("../../data/health/birth_weight.parquet")
        .assign(
            low_birth_weight = lambda x: x[['Menos de 500g', '500 a 999g', '1000 a 1499 g', '1500 a 2499 g']].sum(axis=1) / x["Total"]
                )
        .drop(columns=[
            'mun_name', 'Menos de 500g', '500 a 999g', '1000 a 1499 g', '1500 a 2499 g',
            '2500 a 2999 g', '3000 a 3999 g','4000g e mais', 'Ignorado'
            ])
        .rename(columns={"Total": "total_births"})
    )

birth_weight

,mun_id,year,total_births,low_birth_weight
0,110001,1994,2.0,0.000000
1,110002,1994,8.0,0.250000
2,110004,1994,7.0,0.000000
3,110005,1994,1.0,0.000000
4,110006,1994,6.0,0.000000
...,...,...,...,...
163788,522200,2023,191.0,0.104712
163789,522205,2023,137.0,0.109489
163790,522220,2023,53.0,0.094340
163791,522230,2023,50.0,0.060000


In [4]:
gestational_duration = pd.read_parquet("../../data/health/gestational_duration.parquet")
gestational_duration["preterm_birth"] = gestational_duration[['Menos de 22 semanas', 'De 22 a 27 semanas', 'De 28 a 36 semanas, não especificado', 'De 28 a 31 semanas', 'De 32 a 36 semanas']].sum(axis=1) / gestational_duration["Total"]

gestational_duration

,mun_id,mun_name,year,Menos de 22 semanas,De 22 a 27 semanas,"De 28 a 36 semanas, não especificado",De 37 a 41 semanas,42 semanas ou mais,Ignorado,De 28 a 31 semanas,De 32 a 36 semanas,Total,preterm_birth
0,110001,ALTA FLORESTA D'OESTE,1994,0.0,0.0,0.0,2.0,0.0,0.0,NaN,NaN,2.0,0.000000
1,110002,ARIQUEMES,1994,0.0,0.0,1.0,7.0,0.0,0.0,NaN,NaN,8.0,0.125000
2,110004,CACOAL,1994,0.0,0.0,0.0,5.0,0.0,2.0,NaN,NaN,7.0,0.000000
3,110005,CEREJEIRAS,1994,0.0,0.0,0.0,1.0,0.0,0.0,NaN,NaN,1.0,0.000000
4,110006,COLORADO DO OESTE,1994,0.0,0.0,0.0,6.0,0.0,0.0,NaN,NaN,6.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
163788,522200,VIANOPOLIS,2023,0.0,2.0,NaN,168.0,2.0,0.0,2.0,17.0,191.0,0.109948
163789,522205,VICENTINOPOLIS,2023,0.0,1.0,NaN,105.0,8.0,1.0,5.0,17.0,137.0,0.167883
163790,522220,VILA BOA,2023,0.0,0.0,NaN,49.0,1.0,0.0,0.0,3.0,53.0,0.056604
163791,522230,VILA PROPICIO,2023,0.0,0.0,NaN,49.0,0.0,0.0,0.0,1.0,50.0,0.020000


In [5]:
hospitalizations_total = pd.read_parquet("../../data/health/hospitalizations.parquet")

hospitalizations_total = hospitalizations_total.query("metric_name=='hospitalizations_count'")[["municipality_code", "year", "metric_value"]]
hospitalizations_total = hospitalizations_total.rename(columns={"municipality_code": "mun_id", "metric_value": "hospitalizations"})

hospitalizations_selected = pd.read_parquet("../../data/health/hospitalizations_selected_morbidity_list.parquet")

focus_channels = [
    "water_sanitation_gastrointestinal", 
    "vector_borne_ecological", 
    "renal_urinary_toxic_water", 
    "toxic_poisoning", 
    "perinatal_newborn",
    "leptospirosis_water_exposure",
    "respiratory_air_dust",
    "skin_contact",
    ]

hospitalizations_selected = hospitalizations_selected.\
    query("metric_name=='hospitalizations_count' & morbidity_channel in @focus_channels").\
        groupby(["municipality_code", "year", "morbidity_channel"], as_index=False).metric_value.sum().\
            pivot( #.query()
                index=["municipality_code", "year"],
                columns="morbidity_channel",
                values="metric_value"
            ).\
                rename(columns = lambda x: x + "_hospitalizations").\
                    reset_index().\
                        rename(columns = {"municipality_code": "mun_id"})
                    
hospitalizations = pd.merge(
    hospitalizations_total,
    hospitalizations_selected,
    on = ["mun_id", "year"],
    how="outer"
)

In [6]:
health_indicators = pd.merge(
    birth_weight,
    gestational_duration[["mun_id", "year", "preterm_birth"]],
    on=["mun_id", "year"],
)

health_indicators = pd.merge(
    health_indicators,
    hospitalizations,
    on=["mun_id", "year"],
)

health_indicators = pd.merge(
    health_indicators,
    population[["mun_id", "year", "total_population", "fertile_population"]],
    on=["mun_id", "year"],
)

for indicator in ["hospitalizations"] + [x + "_hospitalizations" for x in focus_channels]:
    health_indicators[indicator + "_1000"] = health_indicators[indicator] / health_indicators["total_population"] * 1000
    health_indicators.drop(columns=indicator, inplace=True)
    

health_indicators = (
    health_indicators
        .assign(
            general_fertility_rate = health_indicators["total_births"] / health_indicators["fertile_population"]
            )
)

In [7]:
land_cover = pd.read_parquet("../../data/land_cover/land_cover_assembled_adm2.parquet")
land_cover["_bucket"] = ((land_cover["bucket"] // 50) * 50).astype(str).str.pad(3, "left", "0")

In [8]:
land_cover = pd.read_parquet("../../data/land_cover/land_cover_assembled_adm2.parquet")
land_cover["_bucket"] = ((land_cover["bucket"] // 50) * 50).astype(str).str.pad(3, "left", "0")

PIXEL_KM2 = 30 * 30 / 1_000_000  # 0.0009 km² per 30m pixel


def agg_land_cover(query: str, bucket_name: str):
    return (
        land_cover
        .query(query)
        .assign(weighted_share=lambda d: d["share"] * d["cnt"])
        .groupby(["mun_id", "year", "land_cover_class"], as_index=False)
        .agg(
            weighted_share=("weighted_share", "sum"),
            cnt=("cnt", "sum"),
        )
        .assign(
            share=lambda d: np.where(
                d["cnt"] > 0,
                d["weighted_share"] / d["cnt"],
                0,
            )
        )
        .assign(_bucket=bucket_name, bucket=0)
        .drop(columns=["weighted_share"])
    )


land_cover_new_bucket_a = agg_land_cover(
    query="0 <= bucket < 200",
    bucket_name="000200",
)

land_cover_new_bucket_b = agg_land_cover(
    query="50 <= bucket < 200",
    bucket_name="050200",
)

# TODO: needs tuning
land_cover_new_bucket_c = agg_land_cover(
    query="0 > bucket >= -50",
    bucket_name="inside",
)

land_cover = (
    pd.concat(
        [
            land_cover,
            land_cover_new_bucket_a,
            land_cover_new_bucket_b,
            land_cover_new_bucket_c,
        ],
        ignore_index=True,
    )
    .query("bucket == 0 & land_cover_class >= 1")
    .assign(
        col=lambda d: (
            "c"
            + d["land_cover_class"].astype(str)
            + "_b"
            + d["_bucket"].astype(str)
        )
    )
    .pivot_table(
        index=["mun_id", "year"],
        columns="col",
        values=["cnt", "share"],
        aggfunc="first",
    )
    .reset_index()
)

land_cover.columns = [
    f"{col}_{val}" if val else col
    for col, val in land_cover.columns
]

cnt_cols = [c for c in land_cover.columns if c.startswith("cnt_")]

land_cover = (
    land_cover
    .assign(
        **{
            c.replace("cnt_", "km2_"): land_cover[c] * PIXEL_KM2
            for c in cnt_cols
        }
    )
    .drop(columns=cnt_cols)
    .sort_values(["mun_id", "year"])
    .assign(
        c1_b000200_shr_diff=lambda d:
            d.groupby("mun_id")["share_c1_b000200"].diff(),
        c1_b000_shr_diff=lambda d:
            d.groupby("mun_id")["share_c1_b000"].diff(),
    )
)

land_cover.head()

,mun_id,year,share_c1_b000,share_c1_b000200,share_c1_b050200,share_c1_binside,share_c2_b000,share_c2_b000200,share_c2_b050200,share_c2_binside,...,km2_c4_b000,km2_c4_b000200,km2_c4_b050200,km2_c4_binside,km2_c5_b000,km2_c5_b000200,km2_c5_b050200,km2_c5_binside,c1_b000200_shr_diff,c1_b000_shr_diff
0,110001,1985,0.894064,0.682631,0.683273,0.777605,0.017111,0.312560,0.318097,0.145130,...,2.6937,7.4439,3.4191,13.5108,0.2061,471.5694,261.9756,285.4071,NaN,NaN
1,110001,1986,0.889935,0.681907,0.683028,0.776989,0.017080,0.296080,0.303133,0.144020,...,3.8673,8.9478,3.6873,19.1547,0.2025,558.5094,302.7294,314.1072,-0.000723,-0.004129
2,110001,1987,0.876530,0.668415,0.668407,0.769338,0.017003,0.305259,0.312795,0.141580,...,3.4110,8.1279,3.2175,16.8597,0.2043,484.3953,239.0679,371.0403,-0.013493,-0.013405
3,110001,1988,0.864400,0.663560,0.664237,0.760944,0.016836,0.299840,0.301785,0.145420,...,2.7801,7.9038,3.4983,15.3783,0.2061,515.4480,312.9570,308.5110,-0.004854,-0.012131
4,110001,1989,0.852536,0.654894,0.655023,0.754393,0.016767,0.317863,0.317139,0.148192,...,3.0510,8.4996,3.8754,15.4809,0.2016,357.6492,214.1046,264.8007,-0.008666,-0.011863


In [9]:
system_ids = pd.read_parquet("../../data/river_network/adm2_dominant_systems.parquet")
system_ids["mun_id"] = system_ids["adm2"].str[0:6]
system_ids.drop(columns=["adm2"], inplace=True)

system_ids

,system_id,mun_id
0,0,110001
1,0,110002
2,0,110003
3,0,110004
4,0,110005
...,...,...
5561,2,522200
5562,2,522205
5563,11,522220
5564,11,522230


---

In [10]:
analysis = pd.merge(
    health_indicators,
    land_cover,
    on=["mun_id", "year"],
)

analysis = pd.merge(
    analysis,
    system_ids,
    on=["mun_id"],
)
analysis["state_id"] = analysis.mun_id.str[:2]

analysis

,mun_id,year,total_births,low_birth_weight,preterm_birth,total_population,fertile_population,hospitalizations_1000,water_sanitation_gastrointestinal_hospitalizations_1000,vector_borne_ecological_hospitalizations_1000,...,km2_c4_b050200,km2_c4_binside,km2_c5_b000,km2_c5_b000200,km2_c5_b050200,km2_c5_binside,c1_b000200_shr_diff,c1_b000_shr_diff,system_id,state_id
0,110001,2000,617.0,0.051864,0.034036,27095,6451,52.334379,3.653811,0.405979,...,5.5521,10.9773,0.1809,205.4646,115.1505,156.4659,-0.004718,-0.018997,0,11
1,110002,2000,2122.0,0.054665,0.047125,76081,19265,48.67181,1.327532,7.097699,...,0.8469,157.5450,1.8189,2.4138,0.2457,43.8444,-0.007523,-0.019705,0,11
2,110003,2000,129.0,0.015504,0.007752,7677,1704,162.954279,20.971734,1.563111,...,12.0141,12.0834,3.0942,21.8736,11.8188,8.3808,-0.010161,-0.010484,0,11
3,110004,2000,1675.0,0.056716,0.035224,75127,19259,79.625168,3.833509,0.465878,...,12.6414,104.9454,6.0867,61.8840,51.3639,18.9765,-0.013256,-0.014074,0,11
4,110005,2000,331.0,0.039275,0.030211,18593,4534,98.477922,9.465928,1.882429,...,0.3348,15.5061,1.8918,2.5272,0.0477,53.3259,-0.014095,-0.013023,0,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133367,522200,2023,191.0,0.104712,0.109948,15307,3365,76.958254,0.0,0.0,...,0.2133,9.0666,2.3481,3.5109,0.2835,3.4416,-0.001447,-0.001102,2,52
133368,522205,2023,137.0,0.109489,0.167883,8948,2044,59.231113,0.0,0.0,...,1.1664,5.9328,2.7054,5.6709,1.1268,12.2715,0.000175,0.000157,2,52
133369,522220,2023,53.0,0.094340,0.056604,4198,953,53.596951,0.0,0.0,...,0.1314,5.5656,0.4374,1.3509,0.0639,6.5871,-0.001821,-0.002420,11,52
133370,522230,2023,50.0,0.060000,0.020000,5928,1180,54.149798,0.0,0.0,...,8.3196,53.1216,5.4585,15.6870,5.4144,22.7997,-0.001142,-0.002749,11,52


---

# Hypothesis 1

- An increase in mining leads to negative downstream health outcomes

In [12]:
inside_controls = " + share_c30_binside + share_c31_binside + share_c40_binside + share_c41_binside"

hosp_all = "hospitalizations_1000 + water_sanitation_gastrointestinal_hospitalizations_1000 + vector_borne_ecological_hospitalizations_1000 + renal_urinary_toxic_water_hospitalizations_1000 + toxic_poisoning_hospitalizations_1000 + perinatal_newborn_hospitalizations_1000 + respiratory_air_dust_hospitalizations_1000 + skin_contact_hospitalizations_1000 + leptospirosis_water_exposure_hospitalizations_1000"
out_all = "low_birth_weight + preterm_birth + general_fertility_rate + " + hosp_all

In [13]:
fitted = pf.feols(
    f"{out_all} ~ km2_c41_b000200 | mun_id + state_id^year",
    vcov = {"CRV1": "system_id"},
    data = analysis,
    weights = "total_population",
)
pf.etable(fitted, coef_fmt="b:.4f* \n (se:.4f)")

GT(_tbl_data=  __index_level_0__ __index_level_1__                     0  \
0              coef   km2_c41_b000200  0.0000 <br> (0.0000)   
1                fe     state_id^year                     x   
2                fe           mun_id                      x   
3             stats      Observations               128,143   
4             stats                R²                 0.468   

                      1                     2                      3  \
0  0.0001 <br> (0.0001)  0.0000 <br> (0.0000)  0.0384* <br> (0.0170)   
1                     x                     x                      x   
2                     x                     x                      x   
3               128,143               128,143                128,088   
4                 0.548                  0.78                  0.716   

                      4                     5                       6  \
0  0.0185 <br> (0.0103)  0.0025 <br> (0.0019)  0.0099** <br> (0.0037)   
1                     x                     x                       x   
2                     x                     x                       x   
3               128,100               128,100                 128,100   
4                 0.607                  0.37                   0.709   

                      7                      8                      9  \
0  0.0011 <br> (0.0006)  -0.0013 <br> (0.0018)  0.0405* <br> (0.0183)   
1                     x                      x                      x   
2                     x                      x                      x   
3               128,100                128,100                128,100   
4                 0.477                  0.595                  0.714   

                        10                     11  
0  0.0025*** <br> (0.0006)  -0.0000 <br> (0.0000)  
1                        x                      x  
2                        x                      x  
3                  128,100                128,100  
4                     0.66                  0.339  , _body=<great_tables._gt_data.Body object at 0x7f19de1dd6d0>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label='(1)', column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label='(2)', column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label='(3)', column_align='center', column_width=None), ColInfo(var='3', type=<ColInfoTypeEnum.default: 1>, column_label='(4)', column_align='center', column_width=None), ColInfo(var='4', type=<ColInfoTypeEnum.default: 1>, column_label='(5)', column_align='center', column_width=None), ColInfo(var='5', type=<ColInfoTypeEnum.default: 1>, column_label='(6)', column_align='center', column_width=None), ColInfo(var='6', type=<ColInfoTypeEnum.default: 1>, column_label='(7)', column_align='center', column_width=None), ColInfo(var='7', type=<ColInfoTypeEnum.default: 1>, column_label='(8)', column_align='center', column_width=None), ColInfo(var='8', type=<ColInfoTypeEnum.default: 1>, column_label='(9)', column_align='center', column_width=None), ColInfo(var='9', type=<ColInfoTypeEnum.default: 1>, column_label='(10)', column_align='center', column_width=None), ColInfo(var='10', type=<ColInfoTypeEnum.default: 1>, column_label='(11)', column_align='center', column_width=None), ColInfo(var='11', type=<ColInfoTypeEnum.default: 1>, column_label='(12)', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7f1a276a4490>, _spanners=Spanners([SpannerInfo(spanner_id='low_birth_weight', spanner_level=1, spanner_label='low_birth_weight', spanner_units=None, spanner_pattern=None, vars=['0'], 

In [13]:
fitted = pf.feols(
    f"{out_all} ~ km2_c41_b000200 {inside_controls} | mun_id + state_id^year",
    vcov = {"CRV1": "system_id"},
    data = analysis,
    weights = "total_population",
)
pf.etable(fitted, coef_fmt="b:.4f* \n (se:.4f)")

GT(_tbl_data=  __index_level_0__  __index_level_1__                         0  \
0              coef    km2_c41_b000200      0.0000 <br> (0.0000)   
1              coef  share_c30_binside     -0.0030 <br> (0.0064)   
2              coef  share_c31_binside      0.0112 <br> (0.0071)   
3              coef  share_c40_binside  -0.1150*** <br> (0.0316)   
4              coef  share_c41_binside     -0.0952 <br> (0.1077)   
5                fe      state_id^year                         x   
6                fe            mun_id                          x   
7             stats       Observations                   128,143   
8             stats                 R²                     0.471   

                         1                         2  \
0     0.0001 <br> (0.0001)      0.0000 <br> (0.0000)   
1    -0.0089 <br> (0.0162)  -0.0291*** <br> (0.0038)   
2   0.0420** <br> (0.0149)      0.0042 <br> (0.0032)   
3  -0.1967** <br> (0.0612)    0.0621** <br> (0.0227)   
4    -0.1642 <br> (0.2271)    0.0992** <br> (0.0381)   
5                        x                         x   
6                        x                         x   
7                  128,143                   128,143   
8                    0.551                     0.785   

                          3                         4                      5  \
0      0.0214 <br> (0.0233)      0.0142 <br> (0.0080)   0.0026 <br> (0.0019)   
1  30.8330*** <br> (7.6552)      1.0976 <br> (1.1084)  -0.5921 <br> (0.5940)   
2    16.2282* <br> (7.7149)     -1.4648 <br> (2.6166)  -0.2575 <br> (0.4758)   
3    73.2080 <br> (39.9524)  45.2680*** <br> (8.7151)  2.6846* <br> (1.0774)   
4   66.4629 <br> (149.9636)   -20.2956 <br> (15.1925)  -5.2722 <br> (4.1024)   
5                         x                         x                      x   
6                         x                         x                      x   
7                   128,088                   128,100                128,100   
8                     0.718                     0.629                  0.372   

                          6                        7                       8  \
0   0.0069*** <br> (0.0010)     0.0007 <br> (0.0005)   -0.0008 <br> (0.0014)   
1   3.0346*** <br> (0.7513)     0.1402 <br> (0.0906)   -0.0814 <br> (0.6025)   
2   -3.3284** <br> (1.1983)   -0.3561* <br> (0.1641)    0.4981 <br> (0.4921)   
3  15.0653*** <br> (2.1326)  2.1234*** <br> (0.3284)  -2.8694* <br> (1.4414)   
4   -18.4150 <br> (12.5589)    -0.4987 <br> (1.2953)   -1.6055 <br> (4.3778)   
5                         x                        x                       x   
6                         x                        x                       x   
7                   128,100                  128,100                 128,100   
8                     0.726                    0.486                   0.596   

                           9                       10                       11  
0     0.0290** <br> (0.0089)  0.0024*** <br> (0.0006)     0.0000 <br> (0.0000)  
1    8.0137*** <br> (1.9475)   0.5389** <br> (0.1780)     0.0264 <br> (0.0173)  
2      -5.0732 <br> (4.2397)     0.4709 <br> (0.2401)   0.0669** <br> (0.0208)  
3  81.9450*** <br> (11.6538)    -0.7099 <br> (0.7949)  -0.2049** <br> (0.0656)  
4    -30.6405 <br> (51.0592)     1.9449 <br> (2.8896)    -0.3918 <br> (0.2415)  
5                          x                        x                        x  
6                          x                        x                        x  
7                    128,100                  128,100                  128,100  
8                      0.733                    0.661                    0.344  , _body=<great_tables._gt_data.Body object at 0x7f1aeddb5a90>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_

In [14]:
fitted = pf.feols(
    f"{out_all} ~ km2_c41_b050200 | mun_id + state_id^year",
    vcov = {"CRV1": "system_id"},
    data = analysis,
    weights = "total_population",
)
pf.etable(fitted, coef_fmt="b:.4f* \n (se:.4f)")

GT(_tbl_data=  __index_level_0__ __index_level_1__                        0  \
0              coef   km2_c41_b050200  0.0001*** <br> (0.0000)   
1                fe     state_id^year                        x   
2                fe           mun_id                         x   
3             stats      Observations                   67,371   
4             stats                R²                    0.498   

                      1                     2                     3  \
0  0.0002 <br> (0.0001)  0.0000 <br> (0.0000)  0.0165 <br> (0.0128)   
1                     x                     x                     x   
2                     x                     x                     x   
3                67,371                67,371                67,338   
4                 0.559                 0.801                 0.728   

                       4                     5                        6  \
0  0.0087* <br> (0.0036)  0.0004 <br> (0.0006)  0.0035*** <br> (0.0006)   
1                      x                     x                        x   
2                      x                     x                        x   
3                 67,328                67,328                   67,328   
4                  0.643                 0.395                    0.711   

                      7                      8                        9  \
0  0.0003 <br> (0.0004)  0.0024* <br> (0.0010)  0.0207*** <br> (0.0057)   
1                     x                      x                        x   
2                     x                      x                        x   
3                67,328                 67,328                   67,328   
4                 0.535                  0.626                    0.754   

                      10                    11  
0  0.0030* <br> (0.0013)  0.0001 <br> (0.0001)  
1                      x                     x  
2                      x                     x  
3                 67,328                67,328  
4                  0.687                 0.375  , _body=<great_tables._gt_data.Body object at 0x7f19afe60790>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label='(1)', column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label='(2)', column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label='(3)', column_align='center', column_width=None), ColInfo(var='3', type=<ColInfoTypeEnum.default: 1>, column_label='(4)', column_align='center', column_width=None), ColInfo(var='4', type=<ColInfoTypeEnum.default: 1>, column_label='(5)', column_align='center', column_width=None), ColInfo(var='5', type=<ColInfoTypeEnum.default: 1>, column_label='(6)', column_align='center', column_width=None), ColInfo(var='6', type=<ColInfoTypeEnum.default: 1>, column_label='(7)', column_align='center', column_width=None), ColInfo(var='7', type=<ColInfoTypeEnum.default: 1>, column_label='(8)', column_align='center', column_width=None), ColInfo(var='8', type=<ColInfoTypeEnum.default: 1>, column_label='(9)', column_align='center', column_width=None), ColInfo(var='9', type=<ColInfoTypeEnum.default: 1>, column_label='(10)', column_align='center', column_width=None), ColInfo(var='10', type=<ColInfoTypeEnum.default: 1>, column_label='(11)', column_align='center', column_width=None), ColInfo(var='11', type=<ColInfoTypeEnum.default: 1>, column_label='(12)', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7f19e29ff6d0>, _spanners=Spanners([SpannerInfo(spanner_id='low_birth_weight', spanner_level=1, spanner_label='low_birth_weight', spanner_units=None, spanner_pattern=

In [15]:
fitted = pf.feols(
    f"{out_all} ~ km2_c41_b050200 {inside_controls} | mun_id + state_id^year",
    vcov = {"CRV1": "system_id"},
    data = analysis,
    weights = "total_population",
)
pf.etable(fitted, coef_fmt="b:.4f* \n (se:.4f)")

GT(_tbl_data=  __index_level_0__  __index_level_1__                        0  \
0              coef    km2_c41_b050200  0.0001*** <br> (0.0000)   
1              coef  share_c30_binside    -0.0042 <br> (0.0062)   
2              coef  share_c31_binside    0.0159* <br> (0.0066)   
3              coef  share_c40_binside    -0.0805 <br> (0.0429)   
4              coef  share_c41_binside    -0.0318 <br> (0.1385)   
5                fe      state_id^year                        x   
6                fe            mun_id                         x   
7             stats       Observations                   67,371   
8             stats                 R²                    0.499   

                         1                         2  \
0     0.0002 <br> (0.0001)     0.0000* <br> (0.0000)   
1    -0.0112 <br> (0.0147)  -0.0368*** <br> (0.0041)   
2  0.0549*** <br> (0.0143)     -0.0058 <br> (0.0045)   
3    -0.1294 <br> (0.1102)    0.0507** <br> (0.0159)   
4    -0.1184 <br> (0.2730)   0.0930*** <br> (0.0208)   
5                        x                         x   
6                        x                         x   
7                   67,371                    67,371   
8                    0.562                     0.805   

                           3                         4                      5  \
0       0.0093 <br> (0.0120)    0.0093** <br> (0.0029)   0.0006 <br> (0.0004)   
1  40.5732*** <br> (10.9959)      2.4402 <br> (1.5149)  -1.1088 <br> (1.0298)   
2    15.3639** <br> (5.3988)     -1.5981 <br> (2.1972)  -0.9676 <br> (0.8732)   
3     18.3506 <br> (21.5347)  38.8477*** <br> (7.8346)   0.9524 <br> (0.9481)   
4     7.4031 <br> (100.7922)   -24.9095 <br> (16.0586)  -4.1407 <br> (5.5425)   
5                          x                         x                      x   
6                          x                         x                      x   
7                     67,338                    67,328                 67,328   
8                       0.73                     0.656                  0.397   

                          6                        7                      8  \
0     0.0029* <br> (0.0013)     0.0003 <br> (0.0003)  0.0024* <br> (0.0011)   
1   4.5142*** <br> (1.1774)     0.0963 <br> (0.1514)  -0.7024 <br> (0.8022)   
2  -2.5251*** <br> (0.7417)   -0.3870* <br> (0.1583)  -0.2130 <br> (0.8660)   
3  19.5571*** <br> (2.5418)  2.0465*** <br> (0.2919)  -4.7146 <br> (2.3793)   
4   -14.5499 <br> (14.4184)     0.3032 <br> (1.6265)  -3.8509 <br> (3.2872)   
5                         x                        x                      x   
6                         x                        x                      x   
7                    67,328                   67,328                 67,328   
8                     0.731                    0.542                  0.628   

                          9                       10                     11  
0   0.0200*** <br> (0.0038)    0.0028* <br> (0.0011)   0.0001 <br> (0.0001)  
1  11.6575*** <br> (3.2224)  0.7641*** <br> (0.1745)   0.0400 <br> (0.0327)  
2     -4.0007 <br> (2.9530)    0.6275* <br> (0.2997)   0.0858 <br> (0.0439)  
3  69.8927*** <br> (9.3645)    -1.1361 <br> (0.8119)  -0.0401 <br> (0.0929)  
4   -39.2213 <br> (45.6606)     0.6567 <br> (2.6293)  -0.1423 <br> (0.1295)  
5                         x                        x                      x  
6                         x                        x                      x  
7                    67,328                   67,328                 67,328  
8                     0.766                    0.688                  0.377  , _body=<great_tables._gt_data.Body object at 0x7f197a3e2e10>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColI

---

# Hypothesis 2
- Pasture reuse causes downstream health problems

In [14]:
fitted = pf.feols(
    f"{out_all} ~ share_c30_b050200 | mun_id + state_id^year",
    vcov = {"CRV1": "system_id"},
    data = analysis,
    weights = "total_population",
)
pf.etable(fitted, coef_fmt="b:.4f* \n (se:.4f)")

GT(_tbl_data=  __index_level_0__  __index_level_1__                      0  \
0              coef  share_c30_b050200  -0.0111 <br> (0.0072)   
1                fe      state_id^year                      x   
2                fe            mun_id                       x   
3             stats       Observations                 67,371   
4             stats                 R²                  0.498   

                       1                         2                        3  \
0  -0.0345 <br> (0.0205)  -0.0175*** <br> (0.0020)  27.3719** <br> (9.6584)   
1                      x                         x                        x   
2                      x                         x                        x   
3                 67,371                    67,371                   67,338   
4                   0.56                     0.802                     0.73   

                      4                     5                       6  \
0  2.5134 <br> (1.9063)  0.0207 <br> (0.2232)  4.3902** <br> (1.3197)   
1                     x                     x                       x   
2                     x                     x                       x   
3                67,328                67,328                  67,328   
4                 0.644                 0.395                   0.719   

                         7                       8                      9  \
0  0.3352*** <br> (0.0701)  -0.8046* <br> (0.3555)  9.8767* <br> (3.9621)   
1                        x                       x                      x   
2                        x                       x                      x   
3                   67,328                  67,328                 67,328   
4                    0.537                   0.627                  0.757   

                     10                     11  
0  0.1253 <br> (0.1582)  -0.0033 <br> (0.0198)  
1                     x                      x  
2                     x                      x  
3                67,328                 67,328  
4                 0.686                  0.375  , _body=<great_tables._gt_data.Body object at 0x7f1abc7f5d50>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label='(1)', column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label='(2)', column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label='(3)', column_align='center', column_width=None), ColInfo(var='3', type=<ColInfoTypeEnum.default: 1>, column_label='(4)', column_align='center', column_width=None), ColInfo(var='4', type=<ColInfoTypeEnum.default: 1>, column_label='(5)', column_align='center', column_width=None), ColInfo(var='5', type=<ColInfoTypeEnum.default: 1>, column_label='(6)', column_align='center', column_width=None), ColInfo(var='6', type=<ColInfoTypeEnum.default: 1>, column_label='(7)', column_align='center', column_width=None), ColInfo(var='7', type=<ColInfoTypeEnum.default: 1>, column_label='(8)', column_align='center', column_width=None), ColInfo(var='8', type=<ColInfoTypeEnum.default: 1>, column_label='(9)', column_align='center', column_width=None), ColInfo(var='9', type=<ColInfoTypeEnum.default: 1>, column_label='(10)', column_align='center', column_width=None), ColInfo(var='10', type=<ColInfoTypeEnum.default: 1>, column_label='(11)', column_align='center', column_width=None), ColInfo(var='11', type=<ColInfoTypeEnum.default: 1>, column_label='(12)', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7f1ab6e6c690>, _spanners=Spanners([SpannerInfo(spanner_id='low_birth_weight', spanner_level=1, spanner_label='low_birth_wei

In [15]:
fitted = pf.feols(
    f"{out_all} ~ share_c30_b050200 {inside_controls} | mun_id + state_id^year",
    vcov = {"CRV1": "system_id"},
    data = analysis,
    weights = "total_population",
)
pf.etable(fitted, coef_fmt="b:.4f* \n (se:.4f)")

GT(_tbl_data=  __index_level_0__  __index_level_1__                      0  \
0              coef  share_c30_b050200  -0.0050 <br> (0.0062)   
1              coef  share_c30_binside  -0.0011 <br> (0.0049)   
2              coef  share_c31_binside  0.0152* <br> (0.0069)   
3              coef  share_c40_binside  -0.0800 <br> (0.0449)   
4              coef  share_c41_binside  -0.0283 <br> (0.1386)   
5                fe      state_id^year                      x   
6                fe            mun_id                       x   
7             stats       Observations                 67,371   
8             stats                 R²                  0.499   

                         1                         2  \
0    -0.0180 <br> (0.0169)    -0.0054* <br> (0.0025)   
1     0.0001 <br> (0.0135)  -0.0334*** <br> (0.0043)   
2  0.0527*** <br> (0.0139)     -0.0065 <br> (0.0045)   
3    -0.1273 <br> (0.1165)    0.0515** <br> (0.0157)   
4    -0.1107 <br> (0.2606)   0.0937*** <br> (0.0214)   
5                        x                         x   
6                        x                         x   
7                   67,371                    67,371   
8                    0.562                     0.805   

                          3                         4                      5  \
0    18.6109* <br> (7.8093)      0.3779 <br> (1.2990)   0.3485 <br> (0.2395)   
1  29.3562*** <br> (6.7716)      2.2348 <br> (1.5719)  -1.3178 <br> (1.0812)   
2   17.4488** <br> (5.4087)     -1.5642 <br> (2.1612)  -0.9289 <br> (0.8493)   
3    14.5880 <br> (21.4272)  38.6968*** <br> (7.9661)   0.8786 <br> (0.9863)   
4    17.5262 <br> (99.3350)   -23.8604 <br> (15.8683)  -3.9126 <br> (5.4462)   
5                         x                         x                      x   
6                         x                         x                      x   
7                    67,338                    67,328                 67,328   
8                      0.73                     0.656                  0.397   

                          6                        7                        8  \
0   1.9208*** <br> (0.4019)   0.2145** <br> (0.0768)  -0.6552** <br> (0.2387)   
1   3.3614*** <br> (0.8991)    -0.0325 <br> (0.1455)    -0.3008 <br> (0.7197)   
2   -2.3116** <br> (0.7386)   -0.3632* <br> (0.1565)    -0.2890 <br> (0.8689)   
3  19.1528*** <br> (2.3006)  2.0013*** <br> (0.2756)    -4.6048 <br> (2.4987)   
4   -13.3249 <br> (13.6170)     0.4403 <br> (1.5566)    -3.9501 <br> (3.4977)   
5                         x                        x                        x   
6                         x                        x                        x   
7                    67,328                   67,328                   67,328   
8                     0.732                    0.543                    0.629   

                          9                      10                     11  
0     3.2697* <br> (1.4893)   -0.0448 <br> (0.2258)   0.0034 <br> (0.0156)  
1   9.7321*** <br> (2.6741)  0.7981** <br> (0.2930)   0.0382 <br> (0.0301)  
2     -3.6510 <br> (2.8991)   0.6199* <br> (0.2804)   0.0861 <br> (0.0445)  
3  69.0817*** <br> (9.5305)   -1.1504 <br> (0.8026)  -0.0415 <br> (0.0936)  
4   -35.7473 <br> (43.7595)    0.8966 <br> (2.5775)  -0.1321 <br> (0.1202)  
5                         x                       x                      x  
6                         x                       x                      x  
7                    67,328                  67,328                 67,328  
8                     0.766                   0.688                  0.377  , _body=<great_tables._gt_data.Body object at 0x7f1a4a194cd0>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<C

In [16]:
fitted = pf.feols(
    f"{out_all} ~ share_c30_b000200 | mun_id + state_id^year",
    vcov = {"CRV1": "system_id"},
    data = analysis,
    weights = "total_population",
)
pf.etable(fitted, coef_fmt="b:.4f* \n (se:.4f)")

GT(_tbl_data=  __index_level_0__  __index_level_1__                      0  \
0              coef  share_c30_b000200  -0.0064 <br> (0.0108)   
1                fe      state_id^year                      x   
2                fe            mun_id                       x   
3             stats       Observations                128,143   
4             stats                 R²                  0.468   

                       1                         2                       3  \
0  -0.0267 <br> (0.0228)  -0.0236*** <br> (0.0030)  21.7390 <br> (11.5693)   
1                      x                         x                       x   
2                      x                         x                       x   
3                128,143                   128,143                 128,088   
4                  0.548                     0.782                   0.717   

                      4                      5                       6  \
0  1.6951 <br> (2.7123)  -0.1223 <br> (0.2328)  4.3850** <br> (1.6858)   
1                     x                      x                       x   
2                     x                      x                       x   
3               128,100                128,100                 128,100   
4                 0.607                   0.37                   0.715   

                       7                       8                     9  \
0  0.3228* <br> (0.1405)  -0.7099* <br> (0.3059)  8.6205 <br> (6.0978)   
1                      x                       x                     x   
2                      x                       x                     x   
3                128,100                 128,100               128,100   
4                  0.478                   0.595                 0.715   

                     10                     11  
0  0.1636 <br> (0.2084)  -0.0036 <br> (0.0256)  
1                     x                      x  
2                     x                      x  
3               128,100                128,100  
4                  0.66                  0.339  , _body=<great_tables._gt_data.Body object at 0x7f1a2a7d6c50>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label='(1)', column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label='(2)', column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label='(3)', column_align='center', column_width=None), ColInfo(var='3', type=<ColInfoTypeEnum.default: 1>, column_label='(4)', column_align='center', column_width=None), ColInfo(var='4', type=<ColInfoTypeEnum.default: 1>, column_label='(5)', column_align='center', column_width=None), ColInfo(var='5', type=<ColInfoTypeEnum.default: 1>, column_label='(6)', column_align='center', column_width=None), ColInfo(var='6', type=<ColInfoTypeEnum.default: 1>, column_label='(7)', column_align='center', column_width=None), ColInfo(var='7', type=<ColInfoTypeEnum.default: 1>, column_label='(8)', column_align='center', column_width=None), ColInfo(var='8', type=<ColInfoTypeEnum.default: 1>, column_label='(9)', column_align='center', column_width=None), ColInfo(var='9', type=<ColInfoTypeEnum.default: 1>, column_label='(10)', column_align='center', column_width=None), ColInfo(var='10', type=<ColInfoTypeEnum.default: 1>, column_label='(11)', column_align='center', column_width=None), ColInfo(var='11', type=<ColInfoTypeEnum.default: 1>, column_label='(12)', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7f1a3eb82d10>, _spanners=Spanners([SpannerInfo(spanner_id='low_birth_weight', spanner_level=1, spanner_label='low_birth_weight', spanner_unit

In [17]:
fitted = pf.feols(
    f"{out_all} ~ share_c30_b000200 {inside_controls} | mun_id + state_id^year",
    vcov = {"CRV1": "system_id"},
    data = analysis,
    weights = "total_population",
)
pf.etable(fitted, coef_fmt="b:.4f* \n (se:.4f)")

GT(_tbl_data=  __index_level_0__  __index_level_1__                         0  \
0              coef  share_c30_b000200     -0.0001 <br> (0.0059)   
1              coef  share_c30_binside     -0.0028 <br> (0.0055)   
2              coef  share_c31_binside      0.0111 <br> (0.0071)   
3              coef  share_c40_binside  -0.1151*** <br> (0.0314)   
4              coef  share_c41_binside     -0.0841 <br> (0.1063)   
5                fe      state_id^year                         x   
6                fe            mun_id                          x   
7             stats       Observations                   128,143   
8             stats                 R²                     0.471   

                         1                         2                        3  \
0    -0.0122 <br> (0.0131)     -0.0023 <br> (0.0034)    11.1119 <br> (9.2402)   
1     0.0006 <br> (0.0148)  -0.0274*** <br> (0.0038)  22.6845** <br> (7.8049)   
2   0.0404** <br> (0.0149)      0.0039 <br> (0.0033)   16.9829* <br> (7.3906)   
3  -0.1975** <br> (0.0616)    0.0619** <br> (0.0226)   73.8684 <br> (41.0368)   
4    -0.1154 <br> (0.2027)    0.1123** <br> (0.0386)  84.4010 <br> (147.2901)   
5                        x                         x                        x   
6                        x                         x                        x   
7                  128,143                   128,143                  128,088   
8                    0.551                     0.785                    0.718   

                          4                      5                         6  \
0      0.5447 <br> (0.9843)   0.4725 <br> (0.3502)   2.1274*** <br> (0.3963)   
1      0.7412 <br> (1.3188)  -0.9329 <br> (0.7899)     1.4838* <br> (0.6077)   
2     -1.4922 <br> (2.6259)  -0.2339 <br> (0.4609)   -3.1975** <br> (1.1905)   
3  45.2928*** <br> (8.7716)  2.7118* <br> (1.1383)  15.1903*** <br> (2.0561)   
4   -12.8932 <br> (15.2464)  -3.6501 <br> (4.3432)   -13.6052 <br> (11.6695)   
5                         x                      x                         x   
6                         x                      x                         x   
7                   128,100                128,100                   128,100   
8                     0.629                  0.372                     0.727   

                         7                        8  \
0    0.1986* <br> (0.0926)  -1.0729** <br> (0.3586)   
1    -0.0043 <br> (0.0970)     0.7095 <br> (0.7285)   
2   -0.3442* <br> (0.1685)     0.4187 <br> (0.4653)   
3  2.1351*** <br> (0.3440)   -2.9340* <br> (1.4152)   
4    -0.0178 <br> (1.3504)    -2.6874 <br> (4.0096)   
5                        x                        x   
6                        x                        x   
7                  128,100                  128,100   
8                    0.487                    0.596   

                           9                     10                       11  
0       1.5787 <br> (1.9097)  -0.1641 <br> (0.3892)     0.0035 <br> (0.0174)  
1     6.9414** <br> (2.2898)   0.6680 <br> (0.3535)     0.0238 <br> (0.0176)  
2      -5.0936 <br> (4.1876)  0.4465* <br> (0.2246)   0.0671** <br> (0.0209)  
3  82.0239*** <br> (11.7400)  -0.7212 <br> (0.7676)  -0.2047** <br> (0.0649)  
4    -15.1598 <br> (48.3213)   3.0284 <br> (2.7221)    -0.3833 <br> (0.2293)  
5                          x                      x                        x  
6                          x                      x                        x  
7                    128,100                128,100                  128,100  
8                      0.733                  0.661                    0.344  , _body=<great_tables._gt_data.Body object at 0x7f19f08eba50>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', colu